# 02 · Where and when

Phase 2 asks the descriptive questions before any modeling: how many people are dying, how that has changed, where the deaths concentrate, and when they happen.

All counts use complete years, 2015 through 2025 (see `docs/methodology.md`, D1b). Tract-level numbers below 10 deaths are suppressed before anything is shown or saved.

In [ ]:
import sys
from pathlib import Path

# Notebooks live one folder down, so put the project root on the path for `src` imports
sys.path.append(str(Path.cwd().parent))

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import plots
from src.config import (
    ANALYSIS_END_YEAR, ANALYSIS_START_YEAR, PROCESSED_DIR, REFERENCE_DIR, SUPPRESSION_THRESHOLD,
)

plots.set_style()

In [ ]:
deaths = pd.read_csv(PROCESSED_DIR / "overdose_deaths_tracts.csv", dtype={"GEOID": str},
                     parse_dates=["death_date", "incident_date"])
deaths = deaths[deaths["death_year"].between(ANALYSIS_START_YEAR, ANALYSIS_END_YEAR)]

tract_table = pd.read_csv(PROCESSED_DIR / "tract_table.csv", dtype={"GEOID": str})
tracts = gpd.read_file(REFERENCE_DIR / "cook_tracts.gpkg").merge(tract_table, on="GEOID")

print(f"{len(deaths):,} overdose deaths assigned to Cook tracts, {ANALYSIS_START_YEAR}-{ANALYSIS_END_YEAR}")

## How many, and what changed

Stacking fentanyl-involved deaths against all others shows the total and the fentanyl share in one chart, on one axis.

In [ ]:
by_year = (
    deaths.groupby("death_year")
    .agg(total=("casenumber", "size"), fentanyl=("fentanyl_involved", "sum"))
    .assign(other=lambda df: df["total"] - df["fentanyl"],
            fentanyl_share=lambda df: df["fentanyl"] / df["total"])
)
by_year

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
years = by_year.index

ax.bar(years, by_year["fentanyl"], color=plots.ORANGE, width=0.7, label="Fentanyl involved")
# a thin surface-colored edge keeps a visible gap between the stacked segments
ax.bar(years, by_year["other"], bottom=by_year["fentanyl"], color=plots.GRAY_MID, width=0.7,
       edgecolor=plots.SURFACE, linewidth=2, label="No fentanyl listed")

# Direct-label the fentanyl share on the first, peak, and last years only
peak_year = by_year["total"].idxmax()
for year in [years.min(), peak_year, years.max()]:
    share = by_year.loc[year, "fentanyl_share"]
    ax.text(year, by_year.loc[year, "total"] + 25, f"{share:.0%}\nfentanyl",
            ha="center", va="bottom", fontsize=8, color=plots.TEXT_SECONDARY)

ax.set_title("Overdose deaths in Cook County, by year")
ax.set_ylabel("Deaths")
ax.set_xticks(years)
ax.set_ylim(0, by_year["total"].max() * 1.18)
ax.legend(loc="upper left")
plots.add_source_note(fig, "Source: Cook County Medical Examiner Case Archive. Accidental drug poisonings, "
                           "deaths assigned to a Cook County tract.")
plots.save_figure(fig, "annual_deaths_fentanyl.png")
plt.show()

## Where

Rates are deaths per 100k residents per year, pooled over all eleven years so small tracts aren't driven by one or two deaths. Tracts with fewer than 10 deaths in total are suppressed on the map.

In [ ]:
suppressed = tracts["overdose_deaths"] < SUPPRESSION_THRESHOLD
print(f"{suppressed.sum()} of {len(tracts)} tracts suppressed (under {SUPPRESSION_THRESHOLD} deaths)")

fig, ax = plots.map_axes()
plots.plot_binned_rates(ax, tracts, "overdose_rate_per_100k", bins=[0, 10, 20, 40, 80, 160, 2000],
                        suppressed_mask=suppressed, legend_title="Deaths per 100k per year")
ax.set_title("Overdose death rate by census tract, 2015-2025")
plots.add_source_note(fig, "Source: Cook County Medical Examiner; 2020 Census. "
                           "Tracts with fewer than 10 deaths are suppressed.")
plots.save_figure(fig, "overdose_rate_map.png")
plt.show()

How concentrated is it? Sorting tracts from highest to lowest rate shows what share of all deaths happens in a small share of the population.

In [ ]:
by_rate = tract_table.dropna(subset=["overdose_rate_per_100k"]).sort_values("overdose_rate_per_100k", ascending=False)
by_rate["population_share"] = by_rate["population_2020"].cumsum() / by_rate["population_2020"].sum()
by_rate["death_share"] = by_rate["overdose_deaths"].cumsum() / by_rate["overdose_deaths"].sum()

for population_share in [0.05, 0.10, 0.20]:
    row = by_rate[by_rate["population_share"] >= population_share].iloc[0]
    print(f"The highest-rate tracts holding {population_share:.0%} of residents account for "
          f"{row['death_share']:.0%} of deaths")

In [ ]:
# Community areas are the unit Chicagoans actually talk about, so summarize there too.
# Only Chicago deaths have a community area; suburbs show as missing.
community_areas = (
    deaths.dropna(subset=["chi_commarea"])
    .groupby("chi_commarea")
    .size()
    .sort_values(ascending=False)
    .rename("deaths_2015_2025")
)
community_areas.head(10).to_frame()

## When

`incident_date` is the time the incident was reported or the person was found, which is often not when drugs were used. 2.2% of records are logged at exactly 00:00, a placeholder for an unknown time, so those are left out of the hour chart.

In [ ]:
known_time = deaths.dropna(subset=["incident_date"])
placeholder = (known_time["incident_date"].dt.hour == 0) & (known_time["incident_date"].dt.minute == 0)
known_time = known_time[~placeholder]
print(f"left out {placeholder.sum():,} cases logged at exactly midnight")

by_hour = known_time["incident_hour"].value_counts().sort_index()
share_by_hour = by_hour / by_hour.sum()

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(share_by_hour.index, share_by_hour.values, color=plots.BLUE, width=0.75)
ax.set_title("When overdose deaths are reported, by hour of day")
ax.set_xlabel("Hour the incident was reported (0 = midnight)")
ax.set_ylabel("Share of deaths")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f"{value:.0%}"))
ax.set_xticks(range(0, 24, 2))
plots.add_source_note(fig, "Source: Cook County Medical Examiner. Excludes cases logged at exactly 00:00 "
                           "(unknown time). Reported time is often when someone was found.")
plots.save_figure(fig, "deaths_by_hour.png")
plt.show()

In [ ]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
by_weekday = deaths["death_weekday"].value_counts().reindex(weekday_order)
(by_weekday / by_weekday.mean()).round(3).rename("relative to daily average").to_frame()

In [ ]:
# Who: age and sex of the people who died (counts are countywide, so no suppression needed)
print(deaths["gender"].value_counts(normalize=True).round(3))
print()
print(deaths["age"].describe().round(1))
print()
# Median age by year: has the age profile shifted as fentanyl took over?
deaths.groupby("death_year")["age"].median()

## What this shows

- **The fentanyl era, then a decline.** Deaths tripled from 629 in 2015 to a peak of 2,060 in 2022, then fell 56% to 913 in 2025. Fentanyl was listed in 14% of deaths in 2015 and 84% at the 2022 peak. Its share has since dropped to 69% as stimulant involvement keeps rising.
- **Deaths are highly concentrated.** The highest-rate tracts, home to 10% of the county's residents, account for 42% of overdose deaths. Austin alone had 992 deaths over these eleven years, followed by Humboldt Park, West Garfield Park, and North Lawndale.
- **An aging group of people who use drugs.** The median age at death rose steadily from 45 to 53, and 77% of those who died were men. That points toward long-term users rather than new young users, which matters for what kind of treatment outreach fits.
- **Timing reflects when people are found.** Reported incidents peak from late morning through mid-afternoon and are lowest around 3 a.m. Weekends run a few percent above the daily average. Because the recorded time is usually when someone was found, this says more about when deaths are discovered than when drugs were used.

Next: `03_access.ipynb` asks whether treatment supply lines up with where these deaths are happening.